# PyTorch 입문 — 커스텀 Module · 수동 forward

손으로 `y = wx + b`를 쓰고, `nn.Linear`와 `LinearReg`로 같은 식을 감싼 뒤, `model(x)`가 `forward`를 부르는 흐름을 봅니다.  
이론 복습: [`../session08.md`](../session08.md)

> **이 노트북이 하는 일**  
> 수동 forward → `nn.Linear` · `state_dict` → 커스텀 `LinearReg` → `__call__` → 다출력 행렬 · `mse_loss`.

환경: `torch-env` 커널을 쓰는 것을 권장합니다.

## 1. import

In [ ]:
import torch
import numpy as np
from torch import nn
import torch.nn.functional as F

## 2. 손으로 쓰는 forward

스칼라 `w`, `b`로 `yhat = w * x + b`를 계산합니다.

In [ ]:
w = torch.tensor(3.0)  # 0차원 스칼라
b = torch.tensor(-1.0)
print(w, b)

In [ ]:
def forward(x):
    # yhat = wx + b
    yhat = w * x + b
    return yhat

In [ ]:
x = torch.tensor(1.0)
y_forward = forward(x)  # 3*1 - 1 = 2
print(y_forward)
print(y_forward.item())  # 텐서 안 값을 파이썬 숫자로

## 3. 여러 입력에 같은 forward

각 칸에 같은 식을 적용하면 출력 모양이 입력과 같습니다.

In [ ]:
x = torch.tensor([[1.0, 3.0, 5.0]])  # (1, 3)
yhat = forward(x)  # [[2., 8., 14.]]
print(x.size())
print(yhat.shape)

## 4. `nn.Linear` · `state_dict`

입력 1 · 출력 1 직선 층입니다. 생성 직후 가중치는 랜덤입니다.

In [ ]:
from torch.nn import Linear

torch.manual_seed(1)
model = Linear(in_features=1, out_features=1, bias=True)
print(model.state_dict())
print(model.state_dict()["weight"])
print(model.state_dict()["bias"])
print(model.weight)
print(model.bias)
# requires_grad=True : 이 값이 거치는 계산을 기록해 기울기를 구할 수 있음

## 5. 커스텀 Module — `LinearReg`

`nn.Module`을 상속해 `nn.Linear`를 안에 두고, `forward`에서 호출합니다.

In [ ]:
class LinearReg(nn.Module):
    def __init__(self, in_size, out_size):
        # 부모 초기화로 가중치 등록·추적을 켠다
        super().__init__()
        self.linear = nn.Linear(in_size, out_size)

    def forward(self, x):
        out = self.linear(x)  # out = wx + b
        return out

In [ ]:
model = LinearReg(1, 1)
model.state_dict()

## 6. `model(x)` → `__call__` → `forward`

`model(x)`는 `__call__`을 거쳐 `forward`를 실행합니다.

In [ ]:
x = torch.tensor([[1.1], [2.1], [3.1]])
yhat = model(x)  # __call__ → forward
print(yhat)

In [ ]:
class A:
    def __call__(self, x):
        return x + 1


a = A()
print(a(3))

## 7. 짧은 기울기 확인

`requires_grad=True`로 추적을 켠 뒤 `backward`로 `.grad`를 채웁니다.

In [ ]:
x = torch.rand(1, requires_grad=True)
y = torch.rand(1, requires_grad=True)
loss = y - x
loss

In [ ]:
loss.backward()
print(x.grad, y.grad)  # x 편미분 -1, y 편미분 1

## 8. 다출력 행렬 · `mse_loss`

입력 4 · 출력 3이면 `w`는 `(4, 3)`, `b`는 `(3,)` 입니다.

In [ ]:
# y0 = (x0*w00) + (x1*w10) + (x2*w20) + (x3*w30) + b0
# y1 = (x0*w01) + (x1*w11) + (x2*w21) + (x3*w31) + b1
# y2 = (x0*w02) + (x1*w12) + (x2*w22) + (x3*w32) + b2

In [ ]:
x = torch.zeros(4)
y = torch.ones(3)
w = torch.rand(4, 3, requires_grad=True)
b = torch.rand(3, requires_grad=True)
y2 = torch.matmul(x, w) + b
print(b)
print(y2)

In [ ]:
loss = F.mse_loss(y2, y)
loss.backward()
print(w.grad, b.grad, loss)

---

이론 복습: [`../session08.md`](../session08.md) · 다출력 학습: [`09_multi_output_linear.ipynb`](./09_multi_output_linear.ipynb)